<h1>Chapter 9 - Graph RAG: Recipe 1: Basic SLA Graph</h1>
<i>Building knowledge graphs and using them in RAG systems.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch09_graph_rag/9.1_basic_sla_graph.ipynb)

---

This notebook is for Chapter 9 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Install Required Packages

Uncomment and run if needed:
```bash
pip install python-dotenv neo4j
```

In [1]:
!pip install python-dotenv neo4j

  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached neo4j-6.0.3-py3-none-any.whl.metadata (5.2 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
Using cached neo4j-6.0.3-py3-none-any.whl (325 kB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)

   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ------------- -------------------------- 1/3 [python-dotenv]
   ------------- -------------------------- 1/3 [python-dotenv]
   ------------- -------------------------- 1/3 [python-dotenv]
   -------------------------- ------------- 2/3 [neo4j]
   -------------------------- ------------- 2/3 [neo4j]
   -------------------------- ------------- 2/3 [neo4j]
   -------------------------- ------------- 2/3 [neo4j]
   

## Import Libraries

In [6]:
from __future__ import annotations
import argparse
import os
import re
from dataclasses import dataclass
from typing import List
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

True

## Connect to Neo4j

Read the Neo4j connection details from environment variables and create a Neo4j driver instance.

In [7]:
## tag::code_neo4j_sla_connection[]
def get_driver():
    """Create Neo4j driver from environment variables."""
    uri = os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687")
    user = os.getenv("NEO4J_USERNAME", "neo4j")
    pwd = os.getenv("NEO4J_PASSWORD", "testpassword")
    return GraphDatabase.driver(uri, auth=(user, pwd))
## end::code_neo4j_sla_connection[]

## Create Database Constraints

Define and create uniqueness constraints for SLA, Clause, and ClauseType nodes. These constraints prevent duplicate nodes and keep MERGE operations predictable.

In [9]:
## tag::code_neo4j_constraints[]
def create_constraints(driver):
    """Create uniqueness constraints for graph nodes."""
    constraints = [
        "CREATE CONSTRAINT sla_id IF NOT EXISTS " "FOR (s:SLA) REQUIRE s.id IS UNIQUE",
        "CREATE CONSTRAINT clause_id IF NOT EXISTS "
        "FOR (c:Clause) REQUIRE c.id IS UNIQUE",
        "CREATE CONSTRAINT type_name IF NOT EXISTS "
        "FOR (t:ClauseType) REQUIRE t.name IS UNIQUE",
    ]
    with driver.session() as session:
        for constraint in constraints:
            session.run(constraint)


driver = get_driver()
create_constraints(driver)
print("✓ Constraints created")
## end::code_neo4j_constraints[]

ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)

## Define Clause Data Structure and Parser

Parse the SLA markdown file into Clause objects. Each '##' heading is treated as a clause title and the following lines as the clause text.

In [ ]:
# tag::code_sla_chunking[]
@dataclass
class Clause:
    id: str
    title: str
    text: str
    order: int
    clause_type: str = "Other"


def parse_sla_file(path: str, sla_id: str) -> List[Clause]:
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    sections = re.split(r"^##\s+", content, flags=re.MULTILINE)[1:]
    clauses = []

    for idx, section in enumerate(sections, start=1):
        lines = section.strip().splitlines()
        if not lines:
            continue
        title = lines[0].strip()
        text = "\n".join(lines[1:]).strip()

        clauses.append(Clause(id=f"{sla_id}_C{idx}", title=title, text=text, order=idx))

    return clauses
# end::code_sla_chunking[]

## Parse SLA File

Load and parse the SLA document.

In [ ]:
sla_id = "SLA1"
sla_title = "Cloud Compute Service Level Agreement (SLA)"
sla_file_path = "./sample_data/SLA1_SUP1.md"

clauses = parse_sla_file(path=sla_file_path, sla_id=sla_id)
print(f"✓ Parsed {len(clauses)} clauses from {sla_file_path}")

## Write SLA and Clauses to Neo4j

This function writes the SLA and all Clause nodes to Neo4j:
- Creates the SLA node
- Creates Clause nodes and HAS_CLAUSE relationships from the SLA
- Links clauses in order using NEXT relationships

In [ ]:
## tag::write_to_neo4j[]
def write_sla_and_clauses(
    driver, sla_id: str, sla_title: str, clauses: List[Clause]
) -> None:
    with driver.session() as session:
        # SLA node
        session.run(
            """
            MERGE (s:SLA {id: $id})
            SET s.title = $title
            """,
            id=sla_id,
            title=sla_title,
        )

        # Clause nodes + HAS_CLAUSE
        for c in clauses:
            session.run(
                """
                MERGE (cl:Clause {id: $id})
                SET cl.title = $title,
                    cl.text = $text,
                    cl.order = $order
                """,
                id=c.id,
                title=c.title,
                text=c.text,
                order=c.order,
                ctype=c.clause_type,
            )
            session.run(
                """
                MATCH (s:SLA {id: $sla_id})
                MATCH (cl:Clause {id: $cid})
                MERGE (s)-[:HAS_CLAUSE]->(cl)
                """,
                sla_id=sla_id,
                cid=c.id,
            )

        # NEXT relationships to preserve order
        for prev, nxt in zip(clauses, clauses[1:]):
            session.run(
                """
                MATCH (a:Clause {id: $p})
                MATCH (b:Clause {id: $n})
                MERGE (a)-[:NEXT]->(b)
                """,
                p=prev.id,
                n=nxt.id,
            )


write_sla_and_clauses(driver, sla_id, sla_title, clauses)
print("✓ SLA and clauses written to Neo4j")
## end::write_to_neo4j[]

## Infer Clause Types

Add ClauseType to categorize clauses semantically. Use a simple heuristic to map titles like "Availability" or "Support" to semantic types.

In [ ]:
## tag::find_clause_type[]
def infer_clause_type(title: str) -> str:
    """Infer ClauseType based on keywords in the title."""
    title_lower = title.lower()
    keywords = {
        "Availability": ["availability", "uptime"],
        "Support": ["support", "response time", "incident"],
        "Maintenance": ["maintenance"],
        "DataProtection": ["data protection", "gdpr", "privacy"],
        "Liability": ["liability"],
        "Termination": ["termination"],
    }

    for clause_type, words in keywords.items():
        if any(word in title_lower for word in words):
            return clause_type
    return "Other"


for c in clauses:
    c.clause_type = infer_clause_type(c.title)
    
print("✓ Clause types inferred")
## end::find_clause_type[]

## Add ClauseType Nodes and Relationships

Create ClauseType nodes and connect each clause to its type using OF_TYPE relationships.

In [ ]:
## tag::add_clause_types[]
def add_clause_types(session, clauses: List[Clause]):
    # Create ClauseType nodes
    types = [{"clause_type": c.clause_type} for c in clauses]
    session.run(
        """
        UNWIND $rows AS row
        MERGE (t:ClauseType {name: row.clause_type})
    """,
        rows=types,
    )

    # Link clauses to their types
    links = [{"id": c.id, "type": c.clause_type} for c in clauses]
    session.run(
        """
        UNWIND $rows AS row
        MATCH (cl:Clause {id: row.id})
        MATCH (t:ClauseType {name: row.type})
        MERGE (cl)-[:OF_TYPE]->(t)
    """,
        rows=links,
    )


add_clause_types(driver.session(), clauses)
print("✓ ClauseType nodes and relationships added")
## end::add_clause_types[]

## Verify the Graph

Run some Cypher queries to verify that the nodes and relationships were created correctly.

In [ ]:
with driver.session() as session:
    sla_result = session.run("MATCH (s:SLA) RETURN s.id AS id, s.title AS title")
    print("SLAs in the graph:")
    for record in sla_result:
        print(f"- {record['id']}: {record['title']}")

    clause_result = session.run(
        """
        MATCH (s:SLA)-[:HAS_CLAUSE]->(c:Clause)
        RETURN c.id AS id, c.title AS title, c.clause_type AS type
        ORDER BY c.order
        """
    )
    print("\nClauses in the graph:")
    for record in clause_result:
        print(f"- {record['id']}: {record['title']} (Type: {record['type']})")

## Close Connection

In [ ]:
driver.close()
print("✓ Connection closed")